In [11]:
%pip install openpyxl

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


# Làm sạch dữ liệu giá HOSE và gắn nhãn ngành Hàng tiêu dùng thiết yếu

Notebook đọc 2 file lịch sử giá + 1 file danh sách ngành, làm sạch, gắn cờ chất lượng dữ liệu, rồi xuất 3 file vào thư mục `2_DuLieuDaXuLy`.

**Cách chạy:** đặt notebook cùng cấp với thư mục `1_Dulieugoc` (chứa 3 file dữ liệu gốc) rồi chạy lần lượt từ trên xuống. 

## 0. Thư viện


In [12]:
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import display

pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 200)

## 1. Cấu hình đường dẫn, tên cột và các ngưỡng
Khai báo mọi thứ có thể cần chỉnh ở **một chỗ**: đường dẫn file (tương đối so với thư mục chứa notebook), tên cột (nếu nguồn đổi tên cột chỉ sửa ở đây) và các ngưỡng dùng để phân nhóm / gắn cờ. Các ô sau chỉ đọc lại các biến này.

In [ ]:
# --- Đường dẫn ---
DIR_GOC  = Path('data/raw')        # chứa 3 file dữ liệu gốc
DIR_XULY = Path('data/processed/OUTPUT_FILE_1')     # nơi ghi file kết quả (tự tạo nếu chưa có)

FILE_GIA_1 = DIR_GOC / 'lich_su_gia_2024_2025.xlsx'
FILE_GIA_2 = DIR_GOC / 'lich_su_gia_2025_2026.xlsx'
FILE_NGANH = DIR_GOC / 'ds_consumer_staples.csv'

# --- Tên cột ---
COL_CONFIG = {
    'gia': {
        'MA': 'Mã', 'NGAY': 'Ngày',
        'GIA_DONG': 'Giá đóng cửa', 'GIA_DC': 'Giá điều chỉnh', 'THAY_DOI': 'Thay đổi',
        'KL_KHOP': 'Khối lượng khớp lệnh', 'GT_KHOP': 'Giá trị khớp lệnh',
        'KL_TT': 'Khối lượng thỏa thuận', 'GT_TT': 'Giá trị thỏa thuận',
        'GIA_MO': 'Giá mở cửa', 'GIA_CAO': 'Giá cao nhất', 'GIA_THAP': 'Giá thấp nhất',
    },
    'nganh': {
        'MA': 'MÃ CK', 'ISIN': 'MÃ ISIN', 'FIGI': 'MÃ FIGI', 'TEN': 'TÊN DN NIÊM YẾT',
        'KL_DKNY': 'KL ĐĂNG KÝ NIÊM YẾT', 'KL_LH': 'KL LƯU HÀNH', 'NGAY_NY': 'NGÀY NIÊM YẾT',
    },
}
G, N = COL_CONFIG['gia'], COL_CONFIG['nganh']

# Nhóm cột số: cột giá (điền bằng giá phiên trước) và cột khối lượng/giá trị (điền 0)
COT_GIA = [G['GIA_DONG'], G['GIA_DC'], G['GIA_MO'], G['GIA_CAO'], G['GIA_THAP']]
COT_KL  = [G['KL_KHOP'], G['GT_KHOP'], G['KL_TT'], G['GT_TT']]
COT_SO  = COT_GIA + COT_KL

# --- Ngưỡng ---
NGUONG_QUA_NGAN = 60                       # dưới 60 phiên: Qua_ngan
TY_LE_DAY_DU    = 0.9                      # từ 90% số phiên chuẩn trở lên: Day_du
TY_LE_PHU_NGAY  = 0.9                      # ngày có số mã dưới 90% trung vị: Ngay_thieu_du_lieu
NGUONG_BIEN_DO  = 0.075                    # 7% biên độ HOSE + 0,5 điểm dung sai: chỉ gắn cờ, không xóa
NGAY_NOI        = pd.Timestamp('2025-10-20')   # ngày đầu tiên của file giá 2, dùng kiểm tra chỗ nối

## 2. Các hàm dùng chung
Gom các hàm dùng lặp lại ở nhiều ô: `so_tp` (chữ -> số, dấu phẩy là dấu thập phân), `doc_file` (đọc mọi cột ở dạng chữ), `cot_thieu` (trả về danh sách cột khai báo mà file không có), `ghep_gia` (ghép 2 file giá, chuẩn hóa Mã và Ngày) và `bang` (đổi dict "chỉ tiêu -> giá trị" thành DataFrame để hiển thị gọn).

In [14]:
def so_tp(s):
    # '7,1' -> 7.1 ; chữ không đọc được thành số -> NaN
    return pd.to_numeric(s.astype(str).str.strip().str.replace(',', '.', regex=False), errors='coerce')

def doc_file(duong_dan):
    # Đọc mọi cột ở dạng chữ để dấu phẩy thập phân không bị hiểu sai
    if str(duong_dan).lower().endswith(('.xlsx', '.xls')):
        df = pd.read_excel(duong_dan, dtype=str)
    else:
        df = pd.read_csv(duong_dan, dtype=str, sep=None, engine='python', encoding='utf-8-sig')
    df.columns = df.columns.str.strip()
    return df

def cot_thieu(df, cfg):
    # Các cột đã khai báo trong cfg nhưng file không có
    return [v for v in cfg.values() if v not in df.columns]

def ghep_gia(g1, g2):
    # Ghép 2 file giá, bỏ khoảng trắng + viết hoa Mã, đổi Ngày (dd/mm/yyyy) sang kiểu ngày
    df = pd.concat([g1, g2], ignore_index=True)
    df[G['MA']] = df[G['MA']].str.strip().str.upper()
    df['Ngay_dt'] = pd.to_datetime(df[G['NGAY']].str.strip(), format='%d/%m/%Y', errors='coerce')
    return df

def bang(chi_tieu, ten_cot='Giá trị'):
    # {'Tên chỉ tiêu': giá trị, ...} -> DataFrame 1 cột, dùng thay cho nhiều lệnh print
    return pd.Series(chi_tieu, name=ten_cot, dtype=object).to_frame().rename_axis('Chỉ tiêu')

## 3. Đọc file và kiểm tra tên cột
Đọc 2 file giá và file ngành (toàn bộ ở dạng chữ). Bảng kết quả cho biết số dòng, số cột và cột nào trong khai báo (mục 1) bị thiếu, để phát hiện sớm nếu nguồn đổi tên cột. Bên dưới là vài dòng đầu của mỗi nguồn.

In [15]:
gia1  = doc_file(FILE_GIA_1)
gia2  = doc_file(FILE_GIA_2)
nganh = doc_file(FILE_NGANH)

display(pd.DataFrame({
    ten: {'So_dong': df.shape[0], 'So_cot': df.shape[1],
          'Cot_thieu': ', '.join(cot_thieu(df, cfg)) or 'Không thiếu'}
    for ten, df, cfg in [('Giá 1', gia1, G), ('Giá 2', gia2, G), ('Ngành', nganh, N)]
}).T)
display(gia1.head(3))
display(nganh.head(3))

,So_dong,So_cot,Cot_thieu
Giá 1,96809,12,Không thiếu
Giá 2,98461,12,Không thiếu
Ngành,35,7,Không thiếu


,Mã,Ngày,Giá đóng cửa,Giá điều chỉnh,Thay đổi,Khối lượng khớp lệnh,Giá trị khớp lệnh,Khối lượng thỏa thuận,Giá trị thỏa thuận,Giá mở cửa,Giá cao nhất,Giá thấp nhất
0,AAA,17/10/2025,7.94,7.61,"-0,04 (-0,50%)",1350800,10.77,0,0,8,8.04,7.94
1,AAM,17/10/2025,7.2,6.9,"+0,05 (+0,70%)",11500,0.08,0,0,7.15,7.2,7.15
2,AAT,17/10/2025,3.66,3.66,"-0,04 (-1,08%)",13000,0.05,0,0,3.69,3.7,3.66


,MÃ CK,MÃ ISIN,MÃ FIGI,TÊN DN NIÊM YẾT,KL ĐĂNG KÝ NIÊM YẾT,KL LƯU HÀNH,NGÀY NIÊM YẾT
0,AAM,VN000000AAM9,BBG000PDD0V4,Công ty Cổ phần Thủy sản Mekong,"12,346,411","10,451,182",13/07/2009
1,AAN,VN000000AAN7,BBG022JL54J2,Công ty Cổ phần Lương thực A An,"91,649,951","91,649,951",07/05/2026
2,ABT,VN000000ABT2,BBG000QMW140,Công ty Cổ phần Xuất nhập khẩu Thủy sản Bến Tre,"14,387,207","11,777,257",06/12/2006


## 4. Hồ sơ "trước làm sạch" của từng file giá
Bảng 1: mỗi file một dòng với số dòng, số mã, ngày đầu/cuối, số ngày giao dịch, ngày không đọc được và số dòng trùng (Mã + Ngày). Bảng 2: theo từng cột số, số ô trống, số ô có chữ không đọc được thành số và số ô bằng 0.

In [16]:
ho_so, theo_cot = {}, {}
for ten, df in [('Giá 1', gia1), ('Giá 2', gia2)]:
    ngay = pd.to_datetime(df[G['NGAY']].str.strip(), format='%d/%m/%Y', errors='coerce')
    ma = df[G['MA']].str.strip().str.upper()
    so = df[COT_SO].apply(so_tp)          # đổi các cột số một lần, dùng lại cho cả 3 thống kê
    ho_so[ten] = {
        'So_dong': len(df), 'So_ma': ma.nunique(),
        'Ngay_dau': ngay.min().strftime('%d/%m/%Y'), 'Ngay_cuoi': ngay.max().strftime('%d/%m/%Y'),
        'So_ngay_giao_dich': ngay.nunique(), 'Ngay_khong_doc_duoc': int(ngay.isna().sum()),
        'Dong_trung_Ma_Ngay': int(df.duplicated(subset=[G['MA'], G['NGAY']]).sum()),
    }
    theo_cot[ten] = pd.DataFrame({
        'O_trong': df[COT_SO].isna().sum(),
        'Khong_doc_duoc': (df[COT_SO].notna() & so.isna()).sum(),
        'Bang_0': (so == 0).sum(),
    })

display(pd.DataFrame(ho_so).T)
display(pd.concat(theo_cot, names=['File', 'Cột']))

,So_dong,So_ma,Ngay_dau,Ngay_cuoi,So_ngay_giao_dich,Ngay_khong_doc_duoc,Dong_trung_Ma_Ngay
Giá 1,96809,400,21/10/2024,17/10/2025,247,0,0
Giá 2,98461,414,20/10/2025,16/10/2026,245,2,0


O_trong  Khong_doc_duoc  Bang_0
File  Cột                                                   
Giá 1 Giá đóng cửa                 0               0       0
      Giá điều chỉnh               0               0       0
      Giá mở cửa                   0               0       0
      Giá cao nhất                 0               0       0
      Giá thấp nhất                0               0       0
      Khối lượng khớp lệnh         0               0    5394
      Giá trị khớp lệnh            0               0    8468
      Khối lượng thỏa thuận        0               0   86103
      Giá trị thỏa thuận           0               0   86590
Giá 2 Giá đóng cửa                 2               0       0
      Giá điều chỉnh               2               0       0
      Giá mở cửa                   2               0       0
      Giá cao nhất                 2               0       0
      Giá thấp nhất                2               0       0
      Khối lượng khớp lệnh         2               0    7348
      Giá trị khớp lệnh            2               0   11093
      Khối lượng thỏa thuận        2               0   87890
      Giá trị thỏa thuận           2               0   87890

## 5. Ghép hai file và kiểm tra đơn vị cột "Giá trị thỏa thuận"
Ghép 2 file giá và hiển thị tổng quan (số dòng, khoảng ngày, số mã, dòng trùng). Rồi kiểm tra đơn vị của Giá trị thỏa thuận bằng cách tính tay trên các dòng có thỏa thuận:

`Ty_le = GT_thỏa_thuận / (KL_thỏa_thuận × Giá_đóng_cửa / 1.000.000)`

Giá tính theo nghìn đồng nên `KL × Giá` ra nghìn đồng, chia 1e6 để ra tỷ đồng. Trung vị `Ty_le` gần 1 là đơn vị tỷ đồng, gần 1000 là triệu đồng.

In [17]:
gia_all = ghep_gia(gia1, gia2)

display(bang({
    'Tổng số dòng sau ghép': len(gia_all),
    'Ngày đầu': gia_all['Ngay_dt'].min().strftime('%d/%m/%Y'),
    'Ngày cuối': gia_all['Ngay_dt'].max().strftime('%d/%m/%Y'),
    'Tổng số ngày giao dịch khác nhau': gia_all['Ngay_dt'].nunique(),
    'Số mã khác nhau': gia_all[G['MA']].nunique(),
    'Dòng trùng (Mã + Ngày) sau ghép': int(gia_all.duplicated(subset=[G['MA'], 'Ngay_dt']).sum()),
}))

tt = gia_all[so_tp(gia_all[G['KL_TT']]) > 0].copy()
tt['KL']  = so_tp(tt[G['KL_TT']])
tt['GT']  = so_tp(tt[G['GT_TT']])
tt['GIA'] = so_tp(tt[G['GIA_DONG']])
tt['Ty_le'] = tt['GT'] / (tt['KL'] * tt['GIA'] / 1e6)

display(bang({
    'Số dòng có giao dịch thỏa thuận': len(tt),
    'Tỷ lệ GT thực tế / GT tính tay (trung vị)': round(tt['Ty_le'].replace([np.inf, -np.inf], np.nan).median(), 3),
}))
display(tt[[G['MA'], G['NGAY'], G['KL_TT'], G['GT_TT'], G['GIA_DONG'], 'Ty_le']].head(8))

,Giá trị
Chỉ tiêu,
Tổng số dòng sau ghép,195270
Ngày đầu,21/10/2024
Ngày cuối,16/10/2026
Tổng số ngày giao dịch khác nhau,492
Số mã khác nhau,420
Dòng trùng (Mã + Ngày) sau ghép,0


,Giá trị
Chỉ tiêu,
Số dòng có giao dịch thỏa thuận,21275
Tỷ lệ GT thực tế / GT tính tay (trung vị),1.0


,Mã,Ngày,Khối lượng thỏa thuận,Giá trị thỏa thuận,Giá đóng cửa,Ty_le
8,ACG,17/10/2025,73950,2.63,35.45,1.003232
50,CII,17/10/2025,3200000,89.28,30.35,0.919275
67,CTG,17/10/2025,24000,1.28,52.2,1.021711
104,DXG,17/10/2025,220000,4.66,22.6,0.937249
118,FPT,17/10/2025,355000,34.08,88.1,1.089671
125,GEX,17/10/2025,64900,4.22,61.5,1.057286
127,GMD,17/10/2025,450000,30.6,68,1.000000
139,HCM,17/10/2025,160000,4.29,26.5,1.011792


## 6. Đối chiếu 35 mã ngành với dữ liệu giá
Chuẩn hóa mã trong file ngành, tóm tắt mỗi mã trên sàn (số phiên, ngày đầu, ngày cuối) rồi nối (left join) vào danh sách 35 mã ngành. Bảng tổng hợp cho biết mã ngành nào **không có** trong dữ liệu HOSE và mã nào có dưới `NGUONG_QUA_NGAN` phiên (gắn `Du_lieu_ngan`); bảng chi tiết xếp theo số phiên tăng dần.

In [18]:
nganh[N['MA']] = nganh[N['MA']].str.strip().str.upper()

tom_tat = gia_all.groupby(G['MA']).agg(
    So_phien=('Ngay_dt', 'nunique'),
    Ngay_dau=('Ngay_dt', 'min'),
    Ngay_cuoi=('Ngay_dt', 'max')
).reset_index()

kt = nganh[[N['MA'], N['TEN'], N['NGAY_NY']]].merge(
    tom_tat, left_on=N['MA'], right_on=G['MA'], how='left')
kt['Co_trong_du_lieu_HOSE'] = kt['So_phien'].notna()
kt['Du_lieu_ngan'] = kt['So_phien'] < NGUONG_QUA_NGAN

ma_lap = nganh.loc[nganh[N['MA']].duplicated(keep=False), N['MA']].tolist()
display(bang({
    'Số dòng file ngành': len(nganh),
    'Số mã ngành khác nhau': nganh[N['MA']].nunique(),
    'Mã ngành bị lặp': ', '.join(ma_lap) or 'Không có',
    'Tổng số phiên chuẩn (số ngày giao dịch khác nhau)': gia_all['Ngay_dt'].nunique(),
    'Mã ngành CÓ trong dữ liệu HOSE': int(kt['Co_trong_du_lieu_HOSE'].sum()),
    'Mã ngành KHÔNG có': int((~kt['Co_trong_du_lieu_HOSE']).sum()),
    'Mã ngành bị gắn Du_lieu_ngan': int(kt['Du_lieu_ngan'].sum()),
}))
display(kt.drop(columns=[G['MA']]).sort_values('So_phien'))

,Giá trị
Chỉ tiêu,
Số dòng file ngành,35
Số mã ngành khác nhau,35
Mã ngành bị lặp,Không có
Tổng số phiên chuẩn (số ngày giao dịch khác nhau),492
Mã ngành CÓ trong dữ liệu HOSE,35
Mã ngành KHÔNG có,0
Mã ngành bị gắn Du_lieu_ngan,0


,MÃ CK,TÊN DN NIÊM YẾT,NGÀY NIÊM YẾT,So_phien,Ngay_dau,Ngay_cuoi,Co_trong_du_lieu_HOSE,Du_lieu_ngan
1,AAN,Công ty Cổ phần Lương thực A An,07/05/2026,103,2026-05-22,2026-10-16,True,False
15,HPA,Công ty Cổ phần Phát triển Nông nghiệp Hòa Phát,29/01/2026,170,2026-02-06,2026-10-16,True,False
5,ANT,Công ty Cổ phần Rau quả Thực phẩm An Giang,30/12/2025,181,2026-01-20,2026-10-16,True,False
4,AFX,Công ty Cổ phần Xuất nhập khẩu Nông sản Thực p...,14/11/2025,210,2025-12-08,2026-10-16,True,False
22,MCH,Công ty Cổ phần Hàng tiêu dùng Masan,10/12/2025,481,2024-10-21,2026-10-16,True,False
34,VNM,Công ty Cổ phần Sữa Việt Nam,28/12/2005,491,2024-10-21,2026-10-16,True,False
33,VHC,Công ty Cổ phần Vĩnh Hoàn,07/12/2007,491,2024-10-21,2026-10-16,True,False
32,VCF,Công ty Cổ phần VINACAFÉ Biên Hòa,21/01/2011,491,2024-10-21,2026-10-16,True,False
7,BAF,Công ty Cổ phần Nông nghiệp BAF Việt Nam,24/11/2021,492,2024-10-21,2026-10-16,True,False
8,BHN,Tổng Công ty Cổ phần Bia – Rượu – Nước giải kh...,30/12/2016,492,2024-10-21,2026-10-16,True,False


## 7. Kiểm tra phiên thiếu của mã ngành và độ phủ theo ngày
Bảng 1: mã ngành có ít phiên hơn lịch chuẩn, kèm số phiên thiếu và danh sách ngày thiếu (chỉ tính trong khoảng từ ngày đầu đến ngày cuối của chính mã đó; `ngày thiếu = ngày trong lịch chuẩn mà mã không có`).
Bảng 2 và 3: số mã giao dịch mỗi ngày (nhỏ nhất, trung vị, lớn nhất) và 5 ngày có ít mã nhất, để phát hiện ngày toàn sàn thiếu dữ liệu.

In [19]:
lich = pd.DatetimeIndex(sorted(gia_all['Ngay_dt'].dropna().unique()))
TONG_PHIEN = len(lich)

# A. Phiên thiếu của từng mã ngành
thieu_ma = []
for ma in kt.loc[kt['So_phien'] < TONG_PHIEN, N['MA']]:
    d = gia_all.loc[gia_all[G['MA']] == ma, 'Ngay_dt'].dropna()
    thieu = lich[(lich >= d.min()) & (lich <= d.max()) & (~lich.isin(d))]
    thieu_ma.append({'Ma': ma, 'So_phien': len(d), 'So_phien_thieu': len(thieu),
                     'Ngay_thieu': ', '.join(x.strftime('%d/%m/%Y') for x in thieu)})
display(pd.DataFrame(thieu_ma, columns=['Ma', 'So_phien', 'So_phien_thieu', 'Ngay_thieu']))

# B. Số mã giao dịch mỗi ngày
dem = gia_all.dropna(subset=['Ngay_dt']).groupby('Ngay_dt')[G['MA']].nunique()
display(bang({'Nhỏ nhất': dem.min(), 'Trung vị': int(dem.median()), 'Lớn nhất': dem.max()}, 'Số mã mỗi ngày'))
display(dem.nsmallest(5).to_frame('So_ma'))

,Ma,So_phien,So_phien_thieu,Ngay_thieu
0,AAN,103,0,
1,AFX,210,0,
2,ANT,181,0,
3,HPA,170,0,
4,MCH,481,11,"02/12/2024, 03/12/2024, 04/12/2024, 05/12/2024..."
5,VCF,491,1,14/08/2025
6,VHC,491,1,14/08/2025
7,VNM,491,1,14/08/2025


,Số mã mỗi ngày
Chỉ tiêu,
Nhỏ nhất,322
Trung vị,394
Lớn nhất,406


,So_ma
Ngay_dt,
2025-08-14,322
2025-03-17,387
2024-12-02,388
2024-12-03,388
2024-12-04,388


## 8. Kiểm tra chỗ nối hai file bằng ngưỡng biên độ 7,5%
Hai file không có ngày trùng nên không so sánh trực tiếp giá điều chỉnh được; kiểm tra gián tiếp qua lợi nhuận ngày:

`Ret = Giá_điều_chỉnh(t) / Giá_điều_chỉnh(t-1) - 1` (tính riêng từng mã)
`Ty_le_vuot_pct = 100 × số mã có |Ret| > 7,5% / số mã có Ret trong ngày`

Các bảng bên dưới: tổng quan dữ liệu, số dòng vượt từng ngưỡng, so sánh ngày nối (20/10/2025) với trung vị các ngày còn lại, 5 ngày vượt ngưỡng nhiều nhất và các mã vượt ngưỡng đúng ngày nối. Nếu ngày nối không cao hơn các ngày khác thì giá hai file không bị lệch.

In [23]:
gia_all = ghep_gia(gia1, gia2)
ma = gia_all[G['MA']]
display(bang({
    'Tổng số dòng': len(gia_all),
    'Số mã khác nhau': ma.nunique()
}))

gia_all['Gia_dc'] = so_tp(gia_all[G['GIA_DC']])
gia_all = (gia_all.drop_duplicates(subset=[G['MA'], 'Ngay_dt'])
           .sort_values([G['MA'], 'Ngay_dt']).reset_index(drop=True))
gia_all['Ret'] = gia_all['Gia_dc'] / gia_all.groupby(G['MA'])['Gia_dc'].shift(1) - 1

r = gia_all['Ret'].dropna()
display(bang({
    '|Ret| > 7%': int((r.abs() > 0.07).sum()),
    f'|Ret| > {NGUONG_BIEN_DO * 100:g}%': int((r.abs() > NGUONG_BIEN_DO).sum()),
    '|Ret| > 10%': int((r.abs() > 0.10).sum()),
}, 'Số dòng'))

gia_all['Vuot'] = gia_all['Ret'].abs() > NGUONG_BIEN_DO
theo_ngay = (gia_all.dropna(subset=['Ret']).groupby('Ngay_dt')
             .agg(So_ma=('Ret', 'size'), So_vuot=('Vuot', 'sum')))
theo_ngay['Ty_le_vuot_pct'] = 100 * theo_ngay['So_vuot'] / theo_ngay['So_ma']

display(pd.DataFrame({
    f'Ngày nối {NGAY_NOI:%d/%m/%Y}': theo_ngay.loc[NGAY_NOI],
    'Trung vị các ngày còn lại': theo_ngay.drop(NGAY_NOI).median(),
}).T.round(3))
display(theo_ngay.nlargest(5, 'Ty_le_vuot_pct'))
display(gia_all.loc[(gia_all['Ngay_dt'] == NGAY_NOI) & gia_all['Vuot'], [G['MA'], 'Gia_dc', 'Ret']])

,Giá trị
Chỉ tiêu,
Tổng số dòng,195270
Số mã khác nhau,420


,Số dòng
Chỉ tiêu,
|Ret| > 7%,349
|Ret| > 7.5%,81
|Ret| > 10%,46


,So_ma,So_vuot,Ty_le_vuot_pct
Ngày nối 20/10/2025,393.0,0.0,0.0
Trung vị các ngày còn lại,394.0,0.0,0.0


,So_ma,So_vuot,Ty_le_vuot_pct
Ngay_dt,,,
2026-02-02,402,34,8.457711
2025-02-04,392,10,2.551020
2025-08-15,393,7,1.781170
2025-04-10,391,4,1.023018
2025-04-03,390,3,0.769231


,Mã,Gia_dc,Ret


## 9. Làm sạch và xuất dữ liệu (pipeline chính)
Gồm 9 bước nhỏ ở các ô bên dưới, chạy lần lượt từ trên xuống. Mọi con số "trước/sau" được ghi vào biến `nhat_ky` để in ở bước cuối.

### Bước 1. Ghép hai file, chuẩn hóa Mã và Ngày
Ghép file, bỏ khoảng trắng và viết hoa mã (để "vnm" và "VNM " không bị coi là 2 mã), đổi Ngày sang kiểu ngày. Nếu có ngày không đọc được thì dừng để xem lại bằng tay.

In [24]:
nhat_ky = {}

df = ghep_gia(gia1, gia2)
nhat_ky['Số dòng đầu vào (sau ghép)'] = len(df)


### Bước 2. Bỏ dòng trùng
Bỏ dòng trùng theo cặp (Mã, Ngày), giữ dòng xuất hiện đầu tiên.

In [25]:
n0 = len(df)
df = df.drop_duplicates(subset=[G['MA'], 'Ngay_dt'], keep='first')
nhat_ky['Dòng trùng đã bỏ'] = n0 - len(df)

### Bước 3. Đổi sang số và sắp xếp
Đổi các cột giá, khối lượng, giá trị từ chữ sang số (dấu phẩy là dấu thập phân), rồi sắp xếp tăng dần theo Mã và Ngày (cũ -> mới) để các bước dùng giá phiên trước hoạt động đúng.

In [26]:
for c in COT_SO:
    df[c] = so_tp(df[c])
df = df.sort_values([G['MA'], 'Ngay_dt']).reset_index(drop=True)

### Bước 4. Xử lý giá thiếu hoặc bằng 0
Giá là chuỗi liên tục nên ô trống hoặc bằng 0 được điền bằng **giá phiên trước của cùng mã** (không dùng trung bình). Dòng nằm ở đầu chuỗi của mã mà vẫn thiếu giá (không có phiên trước để lấy) thì bỏ.

In [27]:
nhat_ky['Ô giá trống hoặc bằng 0 (trước khi điền)'] = int(df[COT_GIA].isna().sum().sum() + (df[COT_GIA] == 0).sum().sum())
df[COT_GIA] = df[COT_GIA].replace(0, np.nan)
df[COT_GIA] = df.groupby(G['MA'])[COT_GIA].ffill()

n1 = len(df)
df = df.dropna(subset=COT_GIA).reset_index(drop=True)
nhat_ky['Dòng bỏ vì thiếu giá ở đầu chuỗi'] = n1 - len(df)

### Bước 5. Xử lý khối lượng và giá trị thiếu
Ô khối lượng/giá trị trống nghĩa là **không có giao dịch** nên điền 0. Không dùng giá trị phiên trước vì sẽ tạo ra dòng tiền giả.

In [28]:
nhat_ky['Ô khối lượng/giá trị trống đã điền 0'] = int(df[COT_KL].isna().sum().sum())
df[COT_KL] = df[COT_KL].fillna(0)

### Bước 6. Chuẩn hóa bảng ngành
Bảng ngành dùng dấu phẩy làm dấu **ngăn nghìn** (vd `10,451,182`), khác bảng giá (dấu phẩy là dấu thập phân), nên xử lý riêng: bỏ dấu `"` và dấu phẩy rồi đổi sang số; Ngày niêm yết đổi sang kiểu ngày. Số ô không đọc được được ghi vào `nhat_ky`.

In [29]:
ng = nganh.copy()
ng[N['MA']] = ng[N['MA']].str.strip().str.upper()
for c in [N['KL_DKNY'], N['KL_LH']]:
    ng[c] = pd.to_numeric(ng[c].astype(str).str.replace('"', '', regex=False)
                          .str.replace(',', '', regex=False).str.strip(), errors='coerce')
ng[N['NGAY_NY']] = pd.to_datetime(ng[N['NGAY_NY']].astype(str).str.strip(), format='%d/%m/%Y', errors='coerce')
nhat_ky['Bảng ngành: ô KL LƯU HÀNH không đọc được'] = int(ng[N['KL_LH']].isna().sum())
nhat_ky['Bảng ngành: ngày niêm yết không đọc được'] = int(ng[N['NGAY_NY']].isna().sum())

### Bước 7. Gắn nhãn ngành (left join)
Left join bảng giá với bảng ngành theo mã: giữ **mọi mã trên sàn**, mã thuộc ngành tiêu dùng có `Is_Staples = 1`, còn lại bằng 0. Có `assert` kiểm tra join không làm đổi số dòng. Khối lượng niêm yết/lưu hành là giá trị tĩnh nên điền xuôi rồi điền ngược theo từng mã.

In [30]:
n2 = len(df)
cot_ng = [N['MA'], N['TEN'], N['KL_DKNY'], N['KL_LH'], N['NGAY_NY']]
df = df.merge(ng[cot_ng], left_on=G['MA'], right_on=N['MA'], how='left')
assert len(df) == n2, 'Left join làm thay đổi số dòng, cần kiểm tra bảng ngành'
df['Is_Staples'] = df[N['MA']].notna().astype(int)
df = df.drop(columns=[N['MA']])
for c in [N['KL_DKNY'], N['KL_LH']]:
    df[c] = df.groupby(G['MA'])[c].transform(lambda s: s.ffill().bfill())

### Bước 8. Phân loại mã theo số phiên và độ gián đoạn
Với mỗi mã tính:
- `So_phien` = số ngày có dữ liệu
- `Phien_thieu` = (vị trí ngày cuối − vị trí ngày đầu + 1 trong lịch chuẩn) − `So_phien`; `Co_gian_doan = 1` nếu `Phien_thieu > 0`
- `Nhom_du_lieu`: `Qua_ngan` nếu số phiên < 60; `Day_du` nếu số phiên ≥ 90% × tổng phiên chuẩn; còn lại `Niem_yet_moi`. Mã ngành có ngày niêm yết muộn hơn ngày đầu dữ liệu cũng bị xếp `Niem_yet_moi` (trừ `Qua_ngan`).

In [31]:
lich = pd.DatetimeIndex(sorted(df['Ngay_dt'].unique()))
TONG_PHIEN = len(lich)

mk = (df.groupby(G['MA'])['Ngay_dt'].agg(['nunique', 'min', 'max'])
        .rename(columns={'nunique': 'So_phien', 'min': 'Ngay_dau', 'max': 'Ngay_cuoi'}))
mk['Phien_thieu'] = (lich.get_indexer(mk['Ngay_cuoi']) - lich.get_indexer(mk['Ngay_dau']) + 1) - mk['So_phien']
mk['Co_gian_doan'] = (mk['Phien_thieu'] > 0).astype(int)
mk['Nhom_du_lieu'] = np.select(
    [mk['So_phien'] < NGUONG_QUA_NGAN, mk['So_phien'] >= TY_LE_DAY_DU * TONG_PHIEN],
    ['Qua_ngan', 'Day_du'], default='Niem_yet_moi')
mk['Is_Staples'] = mk.index.isin(ng[N['MA']]).astype(int)

# Mã ngành niêm yết muộn hơn ngày đầu dữ liệu: coi là niêm yết mới (kể cả khi đã có nhiều phiên)
mk['Ngay_niem_yet'] = mk.index.map(ng.set_index(N['MA'])[N['NGAY_NY']])
niem_yet_muon = mk['Ngay_niem_yet'] > lich.min()
mk.loc[niem_yet_muon & (mk['Nhom_du_lieu'] != 'Qua_ngan'), 'Nhom_du_lieu'] = 'Niem_yet_moi'

df = df.merge(mk[['So_phien', 'Phien_thieu', 'Co_gian_doan', 'Nhom_du_lieu']],
              left_on=G['MA'], right_index=True, how='left')

### Bước 9. Gắn cờ ngày thiếu dữ liệu và cờ trước ngày niêm yết
- `Ty_le_phu = số mã giao dịch trong ngày / trung vị số mã của mọi ngày`; `Ngay_thieu_du_lieu = 1` nếu `Ty_le_phu < 90%`.
- `Truoc_ngay_niem_yet = 1` nếu ngày của dòng sớm hơn ngày niêm yết trên HOSE (vd MCH trước 10/12/2025). Chỉ gắn cờ, không xóa dòng.

In [32]:
nd = df.groupby('Ngay_dt')[G['MA']].nunique().rename('So_ma').to_frame()
nd['Ty_le_phu'] = nd['So_ma'] / nd['So_ma'].median()
nd['Ngay_thieu_du_lieu'] = (nd['Ty_le_phu'] < TY_LE_PHU_NGAY).astype(int)
df = df.merge(nd[['Ngay_thieu_du_lieu']], left_on='Ngay_dt', right_index=True, how='left')

df['Truoc_ngay_niem_yet'] = (df['Ngay_dt'] < df[N['NGAY_NY']]).astype(int)

### Bước 10. Xuất file
Đổi Ngày sang dạng `yyyy-mm-dd` để Power BI đọc đúng kiểu ngày, đưa Mã và Ngày lên 2 cột đầu, rồi ghi 3 file vào `2_DuLieuDaXuLy`: `du_lieu_sach.csv` (dữ liệu chính), `bao_cao_ma.csv` (tóm tắt theo mã), `bao_cao_ngay.csv` (độ phủ theo ngày).

In [33]:
DIR_XULY.mkdir(parents=True, exist_ok=True)

df[G['NGAY']] = df['Ngay_dt'].dt.strftime('%Y-%m-%d')
df = df.drop(columns=['Ngay_dt'])
cot_dau = [G['MA'], G['NGAY']]
df = df[cot_dau + [c for c in df.columns if c not in cot_dau]]

df.to_csv(DIR_XULY / 'du_lieu_sach.csv', index=False, encoding='utf-8-sig')
mk.reset_index().to_csv(DIR_XULY / 'bao_cao_ma.csv', index=False, encoding='utf-8-sig')
nd.reset_index().to_csv(DIR_XULY / 'bao_cao_ngay.csv', index=False, encoding='utf-8-sig')

### Bước 11. Bảng số liệu "sau làm sạch"
Hiển thị dạng bảng để chép vào tài liệu: (1) nhật ký các bước xử lý (số dòng bỏ, số ô điền...), (2) số mã theo nhóm dữ liệu (mã ngành so với toàn sàn), (3) mã ngành có gián đoạn, (4) số dòng gắn cờ `Truoc_ngay_niem_yet` theo mã, (5) 5 dòng đầu của file kết quả.

In [ ]:
nhat_ky['Số dòng sau làm sạch'] = len(df)
nhat_ky['Số mã sau làm sạch'] = df[G['MA']].nunique()
nhat_ky['Số mã ngành (Is_Staples = 1)'] = int(mk['Is_Staples'].sum())
nhat_ky['Số phiên chuẩn'] = TONG_PHIEN
nhat_ky['Ngày có độ phủ dưới 90%'] = ', '.join(d.strftime('%d/%m/%Y') for d in nd.index[nd['Ngay_thieu_du_lieu'] == 1]) or 'Không có'
nhat_ky['Số dòng Truoc_ngay_niem_yet = 1'] = int(df['Truoc_ngay_niem_yet'].sum())
display(bang(nhat_ky))

display(pd.DataFrame({
    'Mã ngành': mk.loc[mk['Is_Staples'] == 1, 'Nhom_du_lieu'].value_counts(),
    'Toàn sàn': mk['Nhom_du_lieu'].value_counts(),
}).fillna(0).astype(int).rename_axis('Nhom_du_lieu'))

display(mk[(mk['Is_Staples'] == 1) & (mk['Co_gian_doan'] == 1)][['So_phien', 'Phien_thieu']])
display(df[df['Truoc_ngay_niem_yet'] == 1].groupby(G['MA']).size().to_frame('So_dong_truoc_niem_yet'))
display(df.head().T)